# LLM Dashboard – OpenRouter
Chạy từng cell theo thứ tự. Cell cuối sẽ mở giao diện web.

In [ ]:
!pip install -q gradio requests

In [ ]:
from google.colab import userdata
import getpass

try:
    API_KEY = userdata.get('OPENROUTER_API_KEY')
    print('Da lay API key tu Colab Secrets.')
except Exception:
    API_KEY = getpass.getpass('Nhap OpenRouter API key: ')

In [ ]:
import requests

ALLOWED_MODELS = [
    'openrouter/auto',
    'owl-alpha',
    'nvidia/nemotron-3-8b-instruct',
    'poolside/laguna-m1',
]

def chat(model, prompt):
    if not prompt.strip():
        return 'Prompt khong duoc de trong.'
    if model not in ALLOWED_MODELS:
        return 'Model khong hop le.'
    try:
        resp = requests.post(
            'https://openrouter.ai/api/v1/chat/completions',
            headers={
                'Authorization': f'Bearer {API_KEY}',
                'Content-Type': 'application/json',
            },
            json={
                'model': model,
                'messages': [{'role': 'user', 'content': prompt}],
                'max_tokens': 1024,
                'temperature': 0.7,
            },
            timeout=60,
        )
        resp.raise_for_status()
        return resp.json()['choices'][0]['message']['content']
    except requests.exceptions.Timeout:
        return 'Loi: API timeout sau 60 giay.'
    except requests.exceptions.HTTPError as e:
        return f'Loi HTTP {e.response.status_code}: {e.response.text[:300]}'
    except Exception as e:
        return f'Loi: {e}'

In [ ]:
import gradio as gr

with gr.Blocks(title='LLM Dashboard') as demo:
    gr.Markdown('# LLM Dashboard – OpenRouter')
    model_dd = gr.Dropdown(choices=ALLOWED_MODELS, value=ALLOWED_MODELS[0], label='Model')
    prompt_box = gr.Textbox(lines=5, placeholder='Nhap prompt tai day...', label='Prompt')
    send_btn = gr.Button('Gui', variant='primary')
    output_box = gr.Textbox(label='Phan hoi', lines=10, interactive=False)
    send_btn.click(fn=chat, inputs=[model_dd, prompt_box], outputs=output_box)
    prompt_box.submit(fn=chat, inputs=[model_dd, prompt_box], outputs=output_box)

demo.launch(share=True)